# Nemotron Val Set Evaluation (vLLM)

**Version: 260408_072lora_950val_v2**

Re-eval of the 0.72 public notebook LoRA adapter on 950 val samples with **corrected parameters** matching Kaggle evaluation page (confirmed by Ryan Holbrook, Kaggle Staff):
- `max_tokens=7680` (was 3584), `max_model_len=8192` (was 4096), `max_num_seqs=64` (was 128)
- `temperature=0.0` (was 1.0) — deterministic greedy decoding

In [ ]:
# === Setup: install vllm + deps from offline packages ===
import subprocess, sys, os, glob

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Step 1: Uninstall system torch (will use torch from offline packages)
subprocess.run('uv pip uninstall torch torchvision torchaudio', shell=True)
print("System torch uninstalled")

# Step 2: Install vllm + all deps from offline packages to system site-packages
offline_candidates = glob.glob('/kaggle/input/**/offline_packages', recursive=True)
if offline_candidates:
    offline_dir = offline_candidates[0]
    print(f"Installing vllm from: {offline_dir}")
    result = subprocess.run(
        f'pip install --no-index --find-links {offline_dir} vllm',
        shell=True, capture_output=True, text=True
    )
    if result.returncode == 0:
        print("vllm + deps installed successfully")
    else:
        print(f"STDERR: {result.stderr[-500:]}")
        raise RuntimeError("Failed to install vllm")
else:
    raise RuntimeError("No offline packages found!")

# Step 3: Fix protobuf MessageFactory.GetPrototype at the source
# The error comes from descriptor_pool.py calling MessageFactory().GetPrototype()
# In newer protobuf, GetPrototype was removed and GetMessageClass doesn't exist either
# We need to find the actual method name and patch it
import google.protobuf
print(f"protobuf version: {google.protobuf.__version__}")

from google.protobuf import message_factory
mf = message_factory.MessageFactory()
available = [m for m in dir(mf) if not m.startswith('_')]
print(f"MessageFactory methods: {available}")

# Patch: add GetPrototype if missing
if not hasattr(message_factory.MessageFactory, 'GetPrototype'):
    # In protobuf 4.x+, the function is module-level GetMessageClass, not a method
    try:
        from google.protobuf.message_factory import GetMessageClass
        message_factory.MessageFactory.GetPrototype = lambda self, descriptor: GetMessageClass(descriptor)
        print("Patched GetPrototype using module-level GetMessageClass")
    except ImportError:
        # Last resort: implement it from descriptor_pool
        from google.protobuf import reflection
        def _get_prototype(self, descriptor):
            cls = reflection.GeneratedProtocolMessageType(
                descriptor.name, (google.protobuf.message.Message,),
                {'DESCRIPTOR': descriptor, '__module__': None})
            return cls
        message_factory.MessageFactory.GetPrototype = _get_prototype
        print("Patched GetPrototype using reflection fallback")

# Write a sitecustomize.py so vLLM subprocesses also get the patch
site_packages = subprocess.run(
    'python -c "import site; print(site.getsitepackages()[0])"',
    shell=True, capture_output=True, text=True
).stdout.strip()
patch_code = '''
import google.protobuf.message_factory as _mf
if not hasattr(_mf.MessageFactory, "GetPrototype"):
    try:
        from google.protobuf.message_factory import GetMessageClass as _gmc
        _mf.MessageFactory.GetPrototype = lambda self, desc: _gmc(desc)
    except ImportError:
        from google.protobuf import reflection as _ref, message as _msg
        import google.protobuf as _gpb
        def _gp(self, desc):
            return _ref.GeneratedProtocolMessageType(desc.name, (_msg.Message,), {"DESCRIPTOR": desc, "__module__": None})
        _mf.MessageFactory.GetPrototype = _gp
'''
sitecustomize_path = os.path.join(site_packages, 'sitecustomize.py')
# Append to existing or create new
existing = ''
if os.path.exists(sitecustomize_path):
    with open(sitecustomize_path) as f:
        existing = f.read()
if 'GetPrototype' not in existing:
    with open(sitecustomize_path, 'a') as f:
        f.write('\n# protobuf MessageFactory patch for vLLM compatibility\n')
        f.write(patch_code)
    print(f"Wrote protobuf patch to {sitecustomize_path}")
else:
    print("sitecustomize.py already has protobuf patch")

# Verify
import vllm
print(f"vllm {vllm.__version__} ready")
print("Setup complete.")

In [ ]:
import math, re, glob, zipfile
import pandas as pd
import kagglehub

# === Paths ===
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# Find adapter — check for adapter_config.json directly, or inside submission.zip
adapter_configs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if not adapter_configs:
    # Look for submission.zip and extract adapter
    zips = glob.glob("/kaggle/input/**/submission.zip", recursive=True)
    if zips:
        extract_dir = "/tmp/adapter"
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(extract_dir)
        print(f"Extracted adapter from {zips[0]} to {extract_dir}")
        adapter_configs = glob.glob(f"{extract_dir}/**/adapter_config.json", recursive=True)

print(f"Found adapter configs: {adapter_configs}")
assert len(adapter_configs) > 0, "No adapter_config.json found!"
LORA_PATH = os.path.dirname(adapter_configs[0])
print(f"Using adapter: {LORA_PATH}")

# Find val CSV — the huikang-labelled 950 val (has 'huikang_category' = 9 finer categories)
val_candidates = glob.glob("/kaggle/input/**/nemotron-val-huikang-950-260407/260407_val.csv", recursive=True)
if not val_candidates:
    val_candidates = glob.glob("/kaggle/input/**/260407_val.csv", recursive=True)
print(f"Val CSV candidates: {val_candidates}")
assert len(val_candidates) > 0, "Val CSV not found! Check dataset is attached."
VAL_PATH = val_candidates[0]
val_df = pd.read_csv(VAL_PATH)

# Report by huikang_category (9 finer categories). Fall back to 'category' if absent.
HAS_HK = 'huikang_category' in val_df.columns
CAT_COL = 'huikang_category' if HAS_HK else 'category'
print(f"Val set: {len(val_df)} rows | reporting on '{CAT_COL}' (huikang present: {HAS_HK})")
print(val_df[CAT_COL].value_counts())

In [ ]:
# === Competition metric functions (verbatim from live metric/nvidia-nemotron-metric, pulled 2026-05-24) ===
# extract_final_answer uses the LAST '}' (rfind) per the 2026-05-08 metric update,
# so answers containing '}' (e.g. cryptarithm '-}') are extracted correctly.

def extract_final_answer(text):
    if text is None:
        return 'NOT_FOUND'

    # For each \boxed{ occurrence, take everything up to the LAST } before the
    # next \boxed{ (or end of text). Handles answers that contain '}' literally
    # (e.g. \boxed{}52} -> "}52") and nested LaTeX like \boxed{\frac{1}{2}}.
    boxed_starts = list(re.finditer(r'\\boxed\{', text))
    matches = []
    for i, m in enumerate(boxed_starts):
        start = m.end()
        end = boxed_starts[i + 1].start() if i + 1 < len(boxed_starts) else len(text)
        segment = text[start:end]
        last_brace = segment.rfind('}')
        matches.append(segment[:last_brace] if last_brace != -1 else segment)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches:
        return matches[-1]

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'

def verify(stored_answer, predicted):
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()
    if re.fullmatch(r'[01]+', stored_answer):
        return predicted.lower() == stored_answer.lower()
    try:
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()

print("Metric functions loaded (live rfind/last-} extractor).")

In [ ]:
# === Load vLLM engine with LoRA support ===
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

print(f"Model path: {MODEL_PATH}")
print(f"LoRA path: {LORA_PATH}")

# Parameters from Kaggle EVALUATION PAGE (confirmed by Ryan Holbrook, Kaggle Staff)
# Metric code defaults are WRONG: max_tokens=3584, temp=1.0, max_model_len=4096, max_num_seqs=128
# Actual eval values: max_tokens=7680, temp=0.0, max_model_len=8192, max_num_seqs=64
llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=64,
    gpu_memory_utilization=0.85,
    dtype='auto',
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=32,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)

# skip_special_tokens=False to preserve <think> tags in raw_output for inspection
# extract_final_answer() is run on a separate decode with skip_special_tokens=True
# to match the official metric behavior exactly.
# logprobs=1 -> vLLM returns, at each step, the chosen token's logprob together with
# its decoded string (.decoded_token), so we can save (token_text, prob) per token.
sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=7680,
    skip_special_tokens=False,
    logprobs=1,
)

lora_request = LoRARequest('adapter', 1, LORA_PATH)

print("vLLM engine loaded with LoRA support.")
print("Using eval page params: max_tokens=7680, temp=0.0, max_model_len=8192, max_num_seqs=64, logprobs=1")

In [ ]:
# === Build prompts using vLLM's tokenizer ===
METRIC_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

tokenizer = llm.get_tokenizer()

def build_prompt(prompt_text):
    user_content = prompt_text + METRIC_SUFFIX
    try:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except Exception:
        return user_content

prompts = [build_prompt(row['prompt']) for _, row in val_df.iterrows()]
print(f"Built {len(prompts)} prompts")
print(f"Sample prompt (first 300 chars): {prompts[0][:300]}")

In [ ]:
# === Run batch inference with vLLM + LoRA ===
import time, json, math

print(f"Running inference on {len(prompts)} samples...")
t0 = time.time()

outputs = llm.generate(prompts, sampling_params, lora_request=lora_request)

elapsed = time.time() - t0
print(f"Inference done in {elapsed:.1f}s ({len(prompts)/elapsed:.2f} samples/sec)")

# Process results
results = []
for i, output in enumerate(outputs):
    out0 = output.outputs[0]
    # raw_output: TRUE raw stream — decode token_ids with skip_special_tokens=False so
    # </think>, <|im_end|>, etc. are preserved. (vLLM's out0.text strips trailing specials.)
    raw_text = tokenizer.decode(out0.token_ids, skip_special_tokens=False)
    # metric_text: decoded with skip_special_tokens=True (matches official metric)
    metric_text = tokenizer.decode(out0.token_ids, skip_special_tokens=True)
    output_token_len = len(out0.token_ids)
    output_char_len = len(raw_text)
    predicted = extract_final_answer(metric_text)
    ground_truth = str(val_df.iloc[i]['answer'])
    correct = verify(ground_truth, predicted)

    # Per-token (token_text, prob) for the chosen (greedy) token at each step.
    # out0.logprobs is List[Dict[token_id -> Logprob(logprob, decoded_token, ...)]].
    token_probs = []
    if out0.logprobs is not None:
        for tid, step in zip(out0.token_ids, out0.logprobs):
            entry = step.get(tid) if step else None
            if entry is None:
                token_probs.append([None, None])
            else:
                tok = entry.decoded_token
                if tok is None:
                    tok = tokenizer.decode([tid])
                token_probs.append([tok, round(math.exp(entry.logprob), 6)])
    # store as JSON string: [[token, prob], ...]
    token_probs_json = json.dumps(token_probs, ensure_ascii=False)

    row = {
        'id': val_df.iloc[i]['id'],
        'category': val_df.iloc[i]['category'],
        'huikang_category': val_df.iloc[i]['huikang_category'] if HAS_HK else val_df.iloc[i]['category'],
        'prompt': val_df.iloc[i]['prompt'],
        'ground_truth': ground_truth,
        'predicted': predicted,
        'correct': correct,
        'output_token_len': output_token_len,
        'output_char_len': output_char_len,
        'raw_output': raw_text,
        'token_probs': token_probs_json,
    }
    results.append(row)

    status = "OK" if correct else "WRONG"
    print(f"  [{i+1}/{len(prompts)}] {status} cat={row['huikang_category']} gt={ground_truth} pred={predicted} tokens={output_token_len} chars={output_char_len}")

In [ ]:
# === Score results ===
results_df = pd.DataFrame(results)

# Overall accuracy
overall_acc = results_df['correct'].mean()
print(f"\n{'='*60}")
print(f"OVERALL ACCURACY: {overall_acc:.4f} ({results_df['correct'].sum()}/{len(results_df)})")
print(f"{'='*60}")

# Per-category breakdown — by huikang_category (9 finer categories)
print(f"\nPer-huikang_category breakdown:")
print(f"{'huikang_category':<28} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 58)
for cat in sorted(results_df['huikang_category'].unique()):
    cat_df = results_df[results_df['huikang_category'] == cat]
    print(f"{cat:<28} {cat_df['correct'].sum():>8} {len(cat_df):>8} {cat_df['correct'].mean():>10.4f}")

# Also show the official 6-category view for leaderboard continuity
print(f"\nPer-official-category breakdown:")
print(f"{'category':<28} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 58)
for cat in sorted(results_df['category'].unique()):
    cat_df = results_df[results_df['category'] == cat]
    print(f"{cat:<28} {cat_df['correct'].sum():>8} {len(cat_df):>8} {cat_df['correct'].mean():>10.4f}")

# Show sample raw outputs (2 correct + 2 wrong per huikang_category)
print(f"\n{'='*60}")
print("SAMPLE RAW OUTPUTS (first 500 chars)")
print(f"{'='*60}")
for cat in sorted(results_df['huikang_category'].unique()):
    cat_df = results_df[results_df['huikang_category'] == cat]
    print(f"\n--- {cat} ---")
    for label, subset in [("CORRECT", cat_df[cat_df['correct']]), ("WRONG", cat_df[~cat_df['correct']])]:
        for _, row in subset.head(2).iterrows():
            print(f"\n  [{label}] GT={row['ground_truth']} PRED={row['predicted']}")
            print(f"  RAW: {row['raw_output'][:500]}")
            print(f"  ... (total {len(row['raw_output'])} chars)")

In [ ]:
# === Save results ===
results_df.to_csv('/kaggle/working/val_eval_results.csv', index=False)
print(f"\nResults saved to /kaggle/working/val_eval_results.csv")
print(f"Columns: {list(results_df.columns)}")